# T0.4 · Citation tracking and author searches

Looks for works that the T0.2 strings did not retrieve, in three ways:

1. **Backward citations:** references often cited by the non-excluded records that are not in the corpus.
2. **Forward citations:** documents that cite the works of the control set.
3. **Authors:** recent publications of the reference groups.

- **Input:** the Scopus export and `corpus.csv` of T0.2, the decisions of T0.3 and `control_set.csv` of T0.1.
- **Output:** `backward_candidates.csv` and three queries ready to paste into the Scopus advanced search (`query_*.txt`).
- **Full loop:** the export of each query goes into the folder of the same name (`query_backward_citations/`, `query_forward_citations/`, `query_authors/`); then the T0.2 notebook is re-run, which adds the records to the corpus, followed by the T0.3 one.
- **To re-run:** run everything. If the corpus or the screening change, the candidates and the queries are recomputed.

In [1]:
import re, unicodedata, warnings
warnings.filterwarnings("ignore")
from collections import defaultdict
from difflib import SequenceMatcher
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", 110)
T02 = Path("../T0.2-search-and-deduplication")
T03 = Path("../T0.3-screening")
CONTROL = Path("../T0.1-protocol/control_set.csv")
CITATION_THRESHOLD = 5      # a reference is a candidate if at least this many non-excluded records cite it
SINCE = 2020                # author searches cover from this year onwards

def norm(t):
    t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]", "", t)

exports = sorted((T02 / "Scopus").glob("*.csv")) + sorted(Path(".").glob("query_*/*.csv"))
scopus = pd.concat(pd.read_csv(f, dtype=str, encoding="utf-8-sig") for f in exports).fillna("").drop_duplicates("EID").set_index("EID")
corpus = pd.read_csv(T02 / "corpus.csv", dtype=str).fillna("").set_index("id")

# Decision in force: the human one if there is one; otherwise that of the first pass
ai = pd.read_csv(T03 / "screening_ai.csv", dtype=str).fillna("").set_index("id")["decision"]
human = pd.read_csv(T03 / "screening.csv", dtype=str).fillna("").set_index("id")["decision"]
corpus["decision"] = human.where(human != "", ai).reindex(corpus.index).fillna("")
print("Human decisions:", (human != "").sum(), "· from the first pass:", (human == "").sum())
corpus["decision"].value_counts().rename("records").to_frame()

Human decisions: 0 · from the first pass: 2632


,records
decision,
exclude,1890
include,468
doubt,274


## 1. Backward citations

The references of the non-excluded records that have a Scopus entry are walked through (the other databases do not give the reference list in a usable format). Each reference is looked up in the corpus by title; those that are missing are grouped and counted.

In [2]:
index = defaultdict(list)                       # first 25 characters of the title -> corpus titles
for rid, title in corpus["title"].items():
    key = norm(re.sub(r"\[[^\]]*\]\s*$", "", title))
    if len(key) >= 25:
        index[key[:25]].append(key)

seeds = corpus[(corpus["decision"] != "exclude") & (corpus["id_scopus"] != "")]
citing, example = defaultdict(set), {}
total = in_corpus = 0
for rid, eid in seeds["id_scopus"].items():
    for ref in re.findall(r".*?\(\d{4}\)(?:; |$)|.+$", scopus.at[eid, "References"]):
        ref = ref.strip().rstrip(";").strip()
        if not ref:
            continue
        total += 1
        authors, _, rest = ref.partition(", ")
        year = (re.findall(r"\((\d{4})\)$", ref) or [""])[0]
        text = norm(rest)
        if any(text.startswith(c[:40]) for c in index.get(text[:25], [])):
            in_corpus += 1
            continue
        group = (norm(authors.split(";")[0])[:12], year, text[:30])
        citing[group].add(rid)
        example.setdefault(group, ref)

print(f"Seed records with references: {(scopus.loc[seeds['id_scopus'], 'References'] != '').sum()} of {len(seeds)}")
print(f"References read: {total} · already in the corpus: {in_corpus} ({in_corpus / total:.0%})")
print(f"Distinct references outside the corpus: {len(citing)}")

Seed records with references: 532 of 628
References read: 9046 · already in the corpus: 3358 (37%)
Distinct references outside the corpus: 4021


Candidates: those cited by at least `CITATION_THRESHOLD` records. Before taking them as new they are compared with the corpus again in an approximate way (same year, similar title), because references carry typos and title variants.

In [3]:
by_year = defaultdict(list)
for rid, row in corpus.iterrows():
    by_year[row["year"]].append(norm(row["title"]))

def similar_in_corpus(year, rest):
    text = norm(rest)
    for y in (year, str(int(year) - 1), str(int(year) + 1)) if year.isdigit() else (year,):
        for title in by_year.get(y, []):
            if len(title) >= 25 and SequenceMatcher(None, title, text[:len(title)]).ratio() >= 0.85:
                return True
    return False

rows = []
for group, ids in citing.items():
    if len(ids) < CITATION_THRESHOLD:
        continue
    ref = example[group]
    rest = ref.partition(", ")[2]
    if similar_in_corpus(group[1], rest):
        continue
    rows.append({"cited_by": len(ids), "year": group[1], "reference": ref, "citing": " ".join(sorted(ids))})
candidates = pd.DataFrame(rows).sort_values(["cited_by", "year"], ascending=[False, True]).reset_index(drop=True)
candidates.to_csv("backward_candidates.csv", index=False, encoding="utf-8")
print("Candidates:", len(candidates))
candidates[["cited_by", "year", "reference"]].head(40)

Candidates: 99


,cited_by,year,reference
0,32,1987,"Stenbakken G.N.; Souders T.M., “Test point selection and testability measures via QR factorization of line..."
1,21,1989,"Brockman J.B.; Director S.W., Predictive subset testing: Optimizing IC parametric performance testing for ..."
2,20,1999,"Sunter S.; Nagi N., Test metrics for analog parametric faults, To Appear in Proceedings of the 17th IEEE V..."
3,19,1992,"Slamani M.; Kaminska B., Analog circuit fault diagnosis based on sensitivity computation and functional te..."
4,18,1989,"Goldberg D.E., Genetic Algorithms in Search, Optimization and Machine Learning, (1989)"
5,18,1991,"Huss S.D.; Gyurcsik R.S., Optimal ordering of analog integrated circuit tests to minimize test time, Proc...."
6,16,1994,"Liberatore A.; Manetti S.; Piccirilli M.C., A new efficient method for analog circuit testability measurem..."
7,15,2014,"Sunter S.; Jurga K.; Dingenen P.; Vanhooren R., Practical random sampling of potential defects for analog ..."
8,14,2001,"Benner S.; Boroffice O., Optimal production test times through adaptive test programming, Proc. IEEE Int. ..."
9,14,2003,"Czaja Z.; Zielonko R., Fault diagnosis in electronic circuits based on bilinear transformation in 3-D and ..."


Query to bring the candidates from Scopus with their abstract. It uses the first words of each title as an exact phrase, bounded to the year of the reference with a margin of one, so it may miss some that carry typos.

In [4]:
def phrase(ref):
    rest = ref.partition(", ")[2]
    words = re.findall(r"[A-Za-z0-9]+", rest.split(", ")[0])[:7]
    return " ".join(words).lower()

phrases = sorted({(phrase(r), int(y)) for r, y in zip(candidates["reference"], candidates["year"])
                  if len(phrase(r).split()) >= 4 and str(y).isdigit()})
query = " OR ".join(f'( TITLE("{p}") AND PUBYEAR > {y - 2} AND PUBYEAR < {y + 2} )' for p, y in phrases)
Path("query_backward_citations.txt").write_text(query + "\n", encoding="utf-8")
print(len(phrases), "titles in query_backward_citations.txt")

94 titles in query_backward_citations.txt


## 2. Forward citations

Documents that cite the works of the control set that have a Scopus entry. The `citations` column is the Scopus count on the date of the export; the sum is an upper bound, because many documents cite several control works and part of them are already in the corpus.

In [5]:
control = pd.read_csv(CONTROL, dtype=str).fillna("")
by_doi = corpus[corpus["doi"] != ""].reset_index().set_index("doi")
control["corpus_id"] = control["doi"].str.lower().map(by_doi["id"]).fillna("")
control["eid"] = control["doi"].str.lower().map(by_doi["id_scopus"]).fillna("")
with_eid = control[control["eid"] != ""].copy()
with_eid["citations"] = pd.to_numeric(scopus.loc[with_eid["eid"], "Cited by"], errors="coerce").fillna(0).astype(int).values

query = " OR ".join(f"REFEID({e})" for e in with_eid["eid"])
Path("query_forward_citations.txt").write_text(query + "\n", encoding="utf-8")
print(f"Control works with a Scopus entry: {len(with_eid)} of {len(control)} · sum of citations: {with_eid['citations'].sum()}")
print("No Scopus entry (searched by hand):", ", ".join(control.loc[control["eid"] == "", "id"]))
with_eid[["id", "reference", "citations"]]

Control works with a Scopus entry: 23 of 24 · sum of citations: 1141
No Scopus entry (searched by hand): K15


,id,reference,citations
0,K01,"Chakrabarti and Chatterjee, ICCAD 2000",9
1,K02,"Huang and Cheng, ITC 1997",3
2,K03,"Huang and Cheng, TCAS-II 2000",20
3,K04,"Tadeusiewicz and Hałgas, JETTA 2017",5
4,K05,"Brosa and Figueras, ETW 1999",3
5,K06,"Zilch, Lubaszewski and Balen, AICSP 2022",9
6,K07,"Tang et al., IEEE TCAD 2018",35
7,K08,"Spinks et al., JETTA 2004",20
8,K09,"Grzechca, ECCTD 2011",8
9,K10,"Yang et al., EAAI 2026",6


## 3. Author searches

Author identifiers are taken from the Scopus export itself. The table is worth checking: the same author may have several identifiers and a namesake may slip in.

In [6]:
AUTHORS = [   # surname, beginning of the first name
    ("Stratigopoulos", "Har"), ("Chatterjee", "Abh"), ("Sunter", "Ste"),
    ("Manetti", "Ste"), ("Piccirilli", "Mar"), ("Grasso", "Fra"), ("Luchetta", "Ant"),
    ("Starzyk", "Jan"), ("Tadeusiewicz", "Mic"), ("Hałgas", "Sta"),
    ("Golonek", "Tom"), ("Rutkowski", "Jer"), ("Grzechca", "Dam"),
    ("Lubaszewski", "Mar"), ("Balen", "Tia"), ("Czaja", "Zbi"),
]
K_AUTHORS = ["K10", "K11"]      # plus all the authors of these control works

profiles = defaultdict(lambda: {"names": set(), "records": 0})
for names in scopus["Author full names"]:
    for chunk in names.split("; "):
        m = re.match(r"(.+?) \((\d+)\)$", chunk.strip())
        if m:
            profiles[m.group(2)]["names"].add(m.group(1))
            profiles[m.group(2)]["records"] += 1
for p in profiles.values():
    p["name"] = max(p["names"], key=lambda n: (len(n), n))      # longest variant; ties broken alphabetically so the result is stable

rows = []
for surname, first in AUTHORS:
    for au_id, p in profiles.items():
        if any(norm(v.partition(", ")[0]) == norm(surname) and norm(v.partition(", ")[2]).startswith(norm(first)) for v in p["names"]):
            rows.append((p["name"], au_id, p["records"], "list"))
for k in K_AUTHORS:
    eid = control.loc[control["id"] == k, "eid"].item()
    if eid:
        for chunk in scopus.at[eid, "Author full names"].split("; "):
            m = re.match(r"(.+?) \((\d+)\)$", chunk.strip())
            rows.append((profiles[m.group(2)]["name"], m.group(2), profiles[m.group(2)]["records"], k))
authors = pd.DataFrame(rows, columns=["author", "au_id", "records_in_scopus", "origin"]).drop_duplicates("au_id")

no_profile = [a for a, n in AUTHORS if not any(norm(x.partition(",")[0]) == norm(a) for x in authors["author"])]
if no_profile:
    print("No identifier in the export (search by hand):", ", ".join(no_profile))

query = "( " + " OR ".join(f"AU-ID({i})" for i in authors["au_id"]) + f" ) AND PUBYEAR > {SINCE - 1}"
Path("query_authors.txt").write_text(query + "\n", encoding="utf-8")
authors

,author,au_id,records_in_scopus,origin
0,"Stratigopoulos, Haralampos-G. D.",6506132589,72,list
1,"Chatterjee, Abhijit",57211726869,104,list
2,"Sunter, Stephen",7004164904,18,list
3,"Manetti, Stefano",57202041744,44,list
4,"Piccirilli, Maria Christina",35618896900,61,list
5,"Grasso, Francesco",44862110300,91,list
6,"Luchetta, Antonio",7003704621,62,list
7,"Starzyk, Janusz A.",7005928365,25,list
8,"Tadeusiewicz, Michał",56060822200,23,list
9,"Hałgas, Stanisław",6506969027,31,list


## 4. Records added

Corpus records that entered only through tracking, by route and by the screening decision in force.

In [7]:
tracking_only = corpus[~corpus["databases"].str.contains("scopus|wos|ieee")]
print("Records contributed only by tracking:", len(tracking_only))
pd.crosstab(tracking_only["databases"], tracking_only["decision"].replace("", "pending"), margins=True)

Records contributed only by tracking: 750


decision,doubt,exclude,include,All
databases,,,,
author,1,366,2,369
forward_citations,74,264,28,366
forward_citations+author,0,13,2,15
All,75,643,32,750


## Conclusions and notes

- **Forward citations: done.** 692 records cite the control works; 388 were not in the corpus. In the first pass, 30 are proposed as include and 74 as doubt.
- **Authors: done.** 422 records since 2020; 389 were not in the corpus, but only 4 are proposed as include and 1 as doubt. The relevant work of those authors was already retrieved by the strings; the rest are other lines of theirs (hardware security, radiation, machinery diagnosis).
- **K04 retrieved.** Tadeusiewicz and Hałgas (2017) enters through tracking.
- **Backward citations: pending.** There are 99 candidates cited by five or more records. The first one, with 32 citations, is Stenbakken and Souders (1987), control work K15. They still have to be exported from Scopus with `query_backward_citations.txt`.
- **What the candidates reveal.** Thematic gaps of the strings: ordering and compaction of production tests (Brockman and Director; Huss and Gyurcsik; Benner and Boroffice; Drineas and Makris), test metrics and defect sampling (Sunter), and embedded diagnosis with a microcontroller (Czaja). There are also books and method references that will be excluded at screening.
- **Provisional.** The seeds depend on the screening, which for now is only the first pass. When the screening is reviewed this notebook has to be re-run.
- **Second round.** Now that K04 is in Scopus, `query_forward_citations.txt` includes one more control work than when it was exported.

### Notes